In [1]:
# ЯЧЕЙКА 1. Установка и импорты
!pip -q install pandas numpy matplotlib seaborn
import os, io, base64, html, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titlesize": 14, "figure.dpi": 130})
AUTHOR = "Власов Максим Николаевич"
GROUP = "ИСТZS-3401"
DATA_NAME = "wcplayerstatistics2026.csv"
OUT = Path("results_big_data_2")
FIG = OUT / "charts"
FIG.mkdir(parents=True, exist_ok=True)
print("Папка результатов:", OUT.resolve())

Папка результатов: /content/results_big_data_2


In [2]:
# ЯЧЕЙКА 2. Загрузка CSV с компьютера
# При повторном выполнении можно использовать уже загруженный файл.
from google.colab import files
uploaded = files.upload()
assert DATA_NAME in uploaded or Path(DATA_NAME).exists(), f"Загрузите {DATA_NAME}"
print("Файл готов:", DATA_NAME)

Saving wcplayerstatistics2026.csv to wcplayerstatistics2026.csv
Файл готов: wcplayerstatistics2026.csv


In [3]:
# ЯЧЕЙКА 3. Первичный осмотр и аудит качества
raw = pd.read_csv(DATA_NAME, encoding="utf-8-sig", encoding_errors="replace")
print("Строк:", len(raw), "Столбцов:", len(raw.columns))
print("Полных дубликатов:", int(raw.duplicated().sum()))
print("Повторов PlayerID:", int(raw["PlayerID"].duplicated().sum()))
display(raw.head(8))
display(pd.DataFrame({"тип": raw.dtypes.astype(str), "пропусков": raw.isna().sum(), "процент_пропусков": (raw.isna().mean() * 100).round(1)}))
display(raw.describe(include="all").transpose().head(26))

Строк: 1039 Столбцов: 26
Полных дубликатов: 0
Повторов PlayerID: 0


,Rk,Player,Pos,Squad,Age,Club,Born,MP,Starts,Min,...,PKatt,CrdY,CrdR,PlayerID,AstPer90,G+APer90,GlsPer90,G-PKPer90,G+A-PKPer90,SecondPos
0,1,Brenden Aaronson,MF,us United States,25.0,1.eng Leeds United,2000.0,1.0,1.0,76.0,...,0.0,0.0,0.0,5bc43860,0.0,0.0,0.0,0.0,0.0,NaN
1,2,Thelo Aasgaard,MF,no Norway,24.0,1.sct Rangers,2002.0,1.0,1.0,90.0,...,0.0,0.0,0.0,c88a28b9,0.0,1.0,1.0,1.0,1.0,NaN
2,3,Hamza Abdelkarim,FW,eg Egypt,18.0,4.es Barcelona B,2008.0,4.0,0.0,60.0,...,0.0,0.0,0.0,057d8f12,0.0,0.0,0.0,0.0,0.0,NaN
3,4,Hossam Abdelmaguid,DF,eg Egypt,25.0,1.eg Zamalek SC,2001.0,2.0,0.0,59.0,...,0.0,0.0,0.0,accbfc94,0.0,0.0,0.0,0.0,0.0,NaN
4,5,Mohamed Abdelmonem,DF,eg Egypt,27.0,1.fr Nice,1999.0,2.0,1.0,14.0,...,0.0,0.0,0.0,bff15d74,0.0,0.0,0.0,0.0,0.0,NaN
5,6,Ali Abdi,DF,tn Tunisia,32.0,1.fr Nice,1993.0,3.0,3.0,269.0,...,0.0,0.0,0.0,ad7cdb35,0.0,0.0,0.0,0.0,0.0,NaN
6,7,Saud Abdulhamid,DF,sa Saudi Arabia,26.0,1.fr Lens,1999.0,3.0,3.0,269.0,...,0.0,1.0,0.0,caa254e4,0.0,0.0,0.0,0.0,0.0,NaN
7,8,Abdulla Abdullaev,DF,uz Uzbekistan,28.0,1.ae Dibba Al Fujairah,1997.0,2.0,2.0,180.0,...,0.0,0.0,0.0,03053fbf,0.0,0.0,0.0,0.0,0.0,NaN


,тип,пропусков,процент_пропусков
Rk,int64,0,0.0
Player,object,0,0.0
Pos,object,0,0.0
Squad,object,0,0.0
Age,float64,0,0.0
Club,object,3,0.3
Born,float64,0,0.0
MP,float64,0,0.0
Starts,float64,0,0.0
Min,float64,0,0.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Rk,1039.0,NaN,NaN,NaN,520.0,300.077768,1.0,260.5,520.0,779.5,1039.0
Player,1039,1039,Leo Østigård,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pos,1039,4,MF,392,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Squad,1039,48,no Norway,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1039.0,NaN,NaN,NaN,27.463908,4.198173,17.0,24.0,27.0,30.0,41.0
Club,1036,386,1.eng Manchester City,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Born,1039.0,NaN,NaN,NaN,1998.022137,4.196578,1985.0,1995.0,1998.0,2001.0,2008.0
MP,1039.0,NaN,NaN,NaN,3.164581,1.664215,1.0,2.0,3.0,4.0,8.0
Starts,1039.0,NaN,NaN,NaN,2.202117,1.87303,0.0,1.0,2.0,3.0,8.0
Min,1039.0,NaN,NaN,NaN,203.474495,158.361191,1.0,71.0,180.0,288.0,810.0


In [4]:
# ЯЧЕЙКА 4. Очистка: пустые строки, пробелы, повторы, пропуски, числовые значения
# Значение неизвестного клуба отмечаем, а отсутствие второй позиции считаем допустимым.
df = raw.copy()
original_n = len(df)
df = df.dropna(how="all").drop_duplicates()
for c in df.select_dtypes(include=["object"]).columns:
    df[c] = df[c].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
df = df.dropna(subset=["Player", "PlayerID", "Squad"])
df = df.drop_duplicates(subset=["PlayerID"], keep="first")
df["Club"] = df["Club"].fillna("Не указан")
df["SecondPos"] = df["SecondPos"].fillna("Нет")
numeric = ["Age", "Born", "MP", "Starts", "Min", "90s", "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "CrdY", "CrdR", "AstPer90", "G+APer90", "GlsPer90", "G-PKPer90", "G+A-PKPer90"]
for c in numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce")
# Нельзя заменять неизвестные голы нулями: такие строки нужно исследовать отдельно.
required = ["Age", "Min", "MP", "Gls", "Ast", "CrdY", "CrdR"]
before_required = len(df)
df = df.dropna(subset=required)
df = df.loc[(df["Min"] > 0) & (df["Age"].between(15, 50)) & (df["MP"] >= 0)].copy()
for c in ["Gls", "Ast", "CrdY", "CrdR", "Starts", "PK", "PKatt"]:
    df = df.loc[df[c].isna() | (df[c] >= 0)]
# Группы сборных очищаем от кода страны (например, 'fr France' -> 'France').
df["Country"] = df["Squad"].str.replace(r"^[a-z]{2,3}\s+", "", regex=True)
print("До очистки:", original_n, "После:", len(df), "Удалено:", original_n-len(df))
print("Строк без ключевых числовых полей:", before_required-len(df))
print("Клуб 'Не указан':", (df["Club"] == "Не указан").sum())
print("Вторая позиция 'Нет':", (df["SecondPos"] == "Нет").sum())
display(df.head())

До очистки: 1039 После: 1039 Удалено: 0
Строк без ключевых числовых полей: 0
Клуб 'Не указан': 3
Вторая позиция 'Нет': 871


,Rk,Player,Pos,Squad,Age,Club,Born,MP,Starts,Min,...,CrdY,CrdR,PlayerID,AstPer90,G+APer90,GlsPer90,G-PKPer90,G+A-PKPer90,SecondPos,Country
0,1,Brenden Aaronson,MF,us United States,25.0,1.eng Leeds United,2000.0,1.0,1.0,76.0,...,0.0,0.0,5bc43860,0.0,0.0,0.0,0.0,0.0,Нет,United States
1,2,Thelo Aasgaard,MF,no Norway,24.0,1.sct Rangers,2002.0,1.0,1.0,90.0,...,0.0,0.0,c88a28b9,0.0,1.0,1.0,1.0,1.0,Нет,Norway
2,3,Hamza Abdelkarim,FW,eg Egypt,18.0,4.es Barcelona B,2008.0,4.0,0.0,60.0,...,0.0,0.0,057d8f12,0.0,0.0,0.0,0.0,0.0,Нет,Egypt
3,4,Hossam Abdelmaguid,DF,eg Egypt,25.0,1.eg Zamalek SC,2001.0,2.0,0.0,59.0,...,0.0,0.0,accbfc94,0.0,0.0,0.0,0.0,0.0,Нет,Egypt
4,5,Mohamed Abdelmonem,DF,eg Egypt,27.0,1.fr Nice,1999.0,2.0,1.0,14.0,...,0.0,0.0,bff15d74,0.0,0.0,0.0,0.0,0.0,Нет,Egypt


In [5]:
# ЯЧЕЙКА 5. Нормализация и формирование дополнительных метрик
# Показатели за 90 минут полезны для сравнения игроков с разным временем на поле.
df["Goals_per_90"] = (df["Gls"] / df["Min"] * 90).round(3)
df["Assists_per_90"] = (df["Ast"] / df["Min"] * 90).round(3)
df["GoalContrib"] = df["Gls"] + df["Ast"]
df["Contrib_per_90"] = (df["GoalContrib"] / df["Min"] * 90).round(3)
df["Cards_total"] = df["CrdY"] + df["CrdR"]
# Min–max [0;1] по выборке. Не заменяет исходные столбцы.
for c in ["Age", "Min", "Gls", "Ast", "GoalContrib", "Cards_total"]:
    lo, hi = df[c].min(), df[c].max()
    df[c + "_norm"] = ((df[c] - lo) / (hi - lo) if hi > lo else 0.0).round(4)
# Уточнение: показатели на 90 минут по 1-5 минутам нестабильны,
# поэтому отдельный рейтинг строим только по игрокам с >=180 минутами.
qualified = df.loc[df["Min"] >= 180].copy()
df.to_csv(OUT / "players_cleaned_normalized.csv", index=False, encoding="utf-8-sig")
print("Игроков >=180 минут:", len(qualified), "из", len(df))
display(df[["Player", "Age", "Min", "Gls", "Ast", "Goals_per_90", "GoalContrib_norm"]].head(8))

Игроков >=180 минут: 532 из 1039


,Player,Age,Min,Gls,Ast,Goals_per_90,GoalContrib_norm
0,Brenden Aaronson,25.0,76.0,0.0,0.0,0.0,0.0000
1,Thelo Aasgaard,24.0,90.0,1.0,0.0,1.0,0.0714
2,Hamza Abdelkarim,18.0,60.0,0.0,0.0,0.0,0.0000
3,Hossam Abdelmaguid,25.0,59.0,0.0,0.0,0.0,0.0000
4,Mohamed Abdelmonem,27.0,14.0,0.0,0.0,0.0,0.0000
5,Ali Abdi,32.0,269.0,0.0,0.0,0.0,0.0000
6,Saud Abdulhamid,26.0,269.0,0.0,0.0,0.0,0.0000
7,Abdulla Abdullaev,28.0,180.0,0.0,0.0,0.0,0.0000


In [6]:
# ЯЧЕЙКА 6. Сводные таблицы (pandas)
summary = {
    "Игроков": len(df), "Сборных": df["Country"].nunique(),
    "Средний возраст": round(df["Age"].mean(), 1),
    "Всего голов в строках игроков": int(df["Gls"].sum()),
    "Всего голевых передач": int(df["Ast"].sum()),
    "Средние минуты": round(df["Min"].mean(), 1),
}
print(summary)
team = df.groupby("Country", as_index=False).agg(Players=("Player", "count"), Goals=("Gls", "sum"), Assists=("Ast", "sum"), Minutes=("Min", "sum"))
team["Contrib"] = team["Goals"] + team["Assists"]
positions = df.groupby("Pos", as_index=False).agg(Players=("Player", "count"), Goals=("Gls", "sum"), Assists=("Ast", "sum"), AvgAge=("Age", "mean"), Minutes=("Min", "sum"))
positions["GoalsPer90Team"] = positions["Goals"] / positions["Minutes"] * 90
print("Топ сборных по голам:")
display(team.sort_values("Goals", ascending=False).head(10))
print("По позициям:")
display(positions.round(2))
print("Топ игроков по голам:")
display(df.nlargest(10, "Gls")[["Player", "Country", "Pos", "Gls", "Ast", "Min"]])

{'Игроков': 1039, 'Сборных': 48, 'Средний возраст': np.float64(27.5), 'Всего голов в строках игроков': 295, 'Всего голевых передач': 224, 'Средние минуты': np.float64(203.5)}
Топ сборных по голам:


,Country,Players,Goals,Assists,Minutes,Contrib
18,France,22,20.0,18.0,7920.0,38.0
17,England,24,20.0,14.0,8213.0,34.0
1,Argentina,24,18.0,12.0,8879.0,30.0
4,Belgium,23,13.0,10.0,6245.0,23.0
40,Spain,22,13.0,10.0,8250.0,23.0
31,Norway,25,12.0,10.0,6270.0,22.0
19,Germany,23,11.0,10.0,4290.0,21.0
6,Brazil,22,10.0,8.0,4950.0,18.0
28,Morocco,23,10.0,9.0,6270.0,19.0
27,Mexico,25,10.0,7.0,4949.0,17.0


По позициям:


,Pos,Players,Goals,Assists,AvgAge,Minutes,GoalsPer90Team
0,DF,349,31.0,42.0,27.90,76950.0,0.04
1,FW,236,135.0,63.0,27.69,32839.0,0.37
2,GK,62,0.0,0.0,31.06,19260.0,0.00
3,MF,392,129.0,119.0,26.37,82361.0,0.14


Топ игроков по голам:


,Player,Country,Pos,Gls,Ast,Min
586,Kylian Mbappé,France,FW,10.0,4.0,695.0
612,Lionel Messi,Argentina,FW,8.0,4.0,740.0
122,Jude Bellingham,England,MF,7.0,1.0,614.0
373,Erling Haaland,Norway,FW,7.0,0.0,465.0
242,Ousmane Dembélé,France,MF,6.0,2.0,592.0
472,Harry Kane,England,FW,6.0,1.0,652.0
706,Mikel Oyarzabal,Spain,FW,5.0,1.0,601.0
461,Vinicius Júnior,Brazil,FW,4.0,1.0,440.0
752,Julián Quiñones,Mexico,FW,4.0,1.0,410.0
833,Ismaila Sarr,Senegal,MF,4.0,1.0,364.0


In [7]:
# ЯЧЕЙКА 7. Десять графиков: seaborn + matplotlib
chart_info = []
def save_plot(name, title, desc, group):
    plt.tight_layout(pad=1.6)
    plt.savefig(FIG / f"{name}.png", dpi=145, bbox_inches="tight", facecolor="white")
    plt.close()
    chart_info.append({"file": f"charts/{name}.png", "name": title, "desc": desc, "group": group})

# 1. Возраст
plt.figure(figsize=(9, 4.7)); sns.histplot(df["Age"], bins=15, color="#3775c8")
plt.title("1. Возрастная структура игроков"); plt.xlabel("Возраст (лет)"); plt.ylabel("Количество")
save_plot("01_age", "Возраст игроков", "Гистограмма распределения футболистов по возрасту; помогает определить преобладающие возрастные группы.", "Состав участников")

# 2. Позиции
plt.figure(figsize=(8, 4.7)); ax=sns.countplot(data=df, x="Pos", order=["GK", "DF", "MF", "FW"], color="#31a59b")
plt.title("2. Число игроков по амплуа"); plt.xlabel("GK — вратарь, DF — защитник, MF — полузащитник, FW — нападающий");plt.ylabel("Количество")
save_plot("02_positions", "Распределение по позициям", "Сравнение представленности игровых амплуа в наборе данных.", "Состав участников")

# 3. Голы команд
best = team.nlargest(12,"Goals").sort_values("Goals")
plt.figure(figsize=(9, 6)); plt.barh(best["Country"], best["Goals"], color="#467ecb");plt.title("3. 12 сборных с наибольшим числом голов");plt.xlabel("Голы, сумма по игрокам")
save_plot("03_team_goals", "Лидеры среди сборных", "Сумма голов, записанных за футболистами каждой сборной в файле; не обязательно эквивалент официальному протоколу.", "Результативность")

# 4. Топ бомбардиров
best = df.nlargest(12,"Gls").sort_values("Gls")
plt.figure(figsize=(9,6));plt.barh(best["Player"],best["Gls"],color="#29a7a0");plt.title("4. Игроки с наибольшим количеством голов");plt.xlabel("Голы")
save_plot("04_scorers", "Топ бомбардиров", "Абсолютное количество голов. Время на поле здесь не учитывается.", "Результативность")

# 5. Ассисты
best=df.nlargest(12,"Ast").sort_values("Ast")
plt.figure(figsize=(9,6));plt.barh(best["Player"],best["Ast"],color="#8367b6");plt.title("5. Лидеры по голевым передачам");plt.xlabel("Голевые передачи")
save_plot("05_assists", "Топ ассистентов", "Показывает игроков, которые чаще всего создавали голы для партнёров по данным файла.", "Результативность")

# 6. Гол+пас на 90 мин; фильтр
best=qualified.nlargest(12,"Contrib_per_90").sort_values("Contrib_per_90")
plt.figure(figsize=(9,6));plt.barh(best["Player"],best["Contrib_per_90"],color="#e0a34b");plt.title("6. Гол+пас за 90 минут (минимум 180 мин)");plt.xlabel("Результативные действия / 90 минут")
save_plot("06_per90", "Эффективность за 90 минут", "Нормировка на игровое время. Игроки с менее чем 180 минутами исключены из этого рейтинга для уменьшения эффекта малой выборки.", "Результативность")

# 7. Минуты и голы
plt.figure(figsize=(9,5.5));sns.scatterplot(data=df,x="Min",y="Gls",hue="Pos",alpha=.65,s=55);plt.title("7. Связь игрового времени и голов");plt.xlabel("Минуты на поле");plt.ylabel("Голы")
save_plot("07_minutes_goals", "Игровое время и голы", "Диаграмма рассеяния помогает увидеть, что большое число сыгранных минут не гарантирует высокую результативность.", "Взаимосвязи")

# 8. Голы/ассисты по позициям
p=positions.set_index("Pos").reindex(["GK","DF","MF","FW"])
ax=p[["Goals","Assists"]].plot(kind="bar",figsize=(9,5),color=["#467ecb","#31a59b"],rot=0)
plt.title("8. Голы и передачи по позициям");plt.xlabel("Амплуа");plt.ylabel("Сумма");plt.legend(["Голы","Передачи"])
save_plot("08_position_scores", "Вклад амплуа в результат", "Совокупное количество голов и ассистов игроков каждого амплуа. Размер групп разный, поэтому это не рейтинг средней эффективности.", "Взаимосвязи")

# 9. Возраст vs гол+пас
plt.figure(figsize=(9,5.5));sns.boxplot(data=df,x="Pos",y="Age",order=["GK","DF","MF","FW"],color="#a7c5e9",showfliers=True)
plt.title("9. Возраст игроков по игровым позициям");plt.xlabel("Амплуа");plt.ylabel("Возраст, лет")
save_plot("09_age_by_position", "Возраст по амплуа", "Медиана, квартили и возможные экстремальные значения возраста в четырёх игровых группах.", "Взаимосвязи")

# 10. Дисциплина по сборным
cards=df.groupby("Country",as_index=False).agg(Yellow=("CrdY","sum"),Red=("CrdR","sum"))
cards["Total"]=cards["Yellow"]+cards["Red"]
best=cards.nlargest(12,"Total").sort_values("Total")
plt.figure(figsize=(9,6));plt.barh(best["Country"],best["Yellow"],label="Жёлтые",color="#e3b64e");plt.barh(best["Country"],best["Red"],left=best["Yellow"],label="Красные",color="#d76869");plt.title("10. Дисциплинарные карточки по сборным");plt.xlabel("Количество карточек");plt.legend()
save_plot("10_cards", "Карточки команд", "Сумма предупреждений и удалений игроков по сборным; не учитывает количество проведённых каждой сборной матчей.", "Дисциплина")
print(f"Графиков: {len(chart_info)}")
assert len(chart_info)>=8


Графиков: 10


In [8]:
# ЯЧЕЙКА 8. Автоматически вычисляемые выводы по данным
leader = df.sort_values(["Gls", "Ast"], ascending=False).iloc[0]
teamleader = team.sort_values("Goals", ascending=False).iloc[0]
mostpos = df["Pos"].value_counts().idxmax()
percent_no_goal = (df["Gls"] == 0).mean()*100
conclusions = [
    f"Проанализировано {len(df)} строк об игроках {df['Country'].nunique()} сборных. Средний возраст — {df['Age'].mean():.1f} года.",
    f"Самое многочисленное амплуа — {mostpos}: {int((df['Pos']==mostpos).sum())} игроков. Размеры групп различаются, поэтому абсолютные суммы голов требуют контекста.",
    f"Лидер по числу голов — {leader['Player']}: {int(leader['Gls'])} голов. Среди сборных по сумме голов игроков лидирует {teamleader['Country']} ({int(teamleader['Goals'])}).",
    f"Без забитых голов в файле — {percent_no_goal:.1f}% футболистов. Это ожидаемо для статистики, охватывающей защитников, полузащитников и вратарей, а также игроков с разным временем на поле.",
    f"Для сопоставления эффективности рассчитаны действия за 90 минут; в соответствующий график включены только {len(qualified)} игрока с 180+ минутами.",
    "Наблюдаемые различия могут объясняться амплуа, количеством проведённых минут и неодинаковым количеством игр; графики показывают связи, но сами по себе не доказывают причинность.",
    "Ограничения: это срез предоставленного CSV без поматчевой динамики, соперников и подтверждения официального происхождения; выводы не следует распространять за пределы файла."
]
for i,v in enumerate(conclusions,1): print(f"{i}. {v}")


1. Проанализировано 1039 строк об игроках 48 сборных. Средний возраст — 27.5 года.
2. Самое многочисленное амплуа — MF: 392 игроков. Размеры групп различаются, поэтому абсолютные суммы голов требуют контекста.
3. Лидер по числу голов — Kylian Mbappé: 10 голов. Среди сборных по сумме голов игроков лидирует France (20).
4. Без забитых голов в файле — 82.8% футболистов. Это ожидаемо для статистики, охватывающей защитников, полузащитников и вратарей, а также игроков с разным временем на поле.
5. Для сопоставления эффективности рассчитаны действия за 90 минут; в соответствующий график включены только 532 игрока с 180+ минутами.
6. Наблюдаемые различия могут объясняться амплуа, количеством проведённых минут и неодинаковым количеством игр; графики показывают связи, но сами по себе не доказывают причинность.
7. Ограничения: это срез предоставленного CSV без поматчевой динамики, соперников и подтверждения официального происхождения; выводы не следует распространять за пределы файла.
